# PDF Compressor — Backend

1. Rode as 3 células abaixo em ordem.
2. Copie a URL `https://xxxx.trycloudflare.com` que aparece na saída.
3. No site [PDF Compressor](https://xangrybadger.github.io/pdf-compressor/), clique em **Sem API** no header e cole a URL.

A URL muda a cada sessão — você precisa colar novamente se reiniciar o notebook.

In [ ]:
!pip install -q fastapi uvicorn pypdf Pillow python-multipart img2pdf

In [ ]:
import os
os.makedirs('app', exist_ok=True)

In [ ]:
%%writefile app/main.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pypdf import PdfReader, PdfWriter
import io
import os

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/api/compress")
async def compress_pdf(file: UploadFile = File(...)):
    try:
        content = await file.read()
        input_stream = io.BytesIO(content)

        reader = PdfReader(input_stream)
        writer = PdfWriter()

        for page in reader.pages:
            page.compress_content_streams()
            writer.add_page(page)

        output_stream = io.BytesIO()
        writer.write(output_stream)
        output_stream.seek(0)

        return StreamingResponse(
            output_stream,
            media_type="application/pdf",
            headers={"Content-Disposition": f'attachment; filename="compressed_{file.filename}"'},
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "PDF Compressor API"}

In [ ]:
import subprocess, threading, time, re

!command -v cloudflared >/dev/null 2>&1 || curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

def run():
    import uvicorn
    uvicorn.run('app.main:app', host='0.0.0.0', port=8002)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

time.sleep(3)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8002'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

url = None
for line in proc.stdout:
    decoded = line.decode(errors='replace')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', decoded)
    if m:
        url = m.group(0)
        break

if url:
    print(f'\n{"="*60}')
    print(f' PDF Compressor Backend Online!')
    print(f' URL: {url}')
    print(f'{"="*60}')
    print(f'\n Cole esta URL no header do site:')
    print(f' https://xangrybadger.github.io/pdf-compressor/')
    print()
else:
    print('Aguardando URL do cloudflared...')
    print('Verifique a saída acima.')